# Comment Category Prediction Challenge: Deployment & Pipeline Export
---
**Core Methodology:** Stacked Ensemble of LightGBM, Logistic Regressions, NB-SVM, and LinearSVC models with explicit specialist binary classifiers.
This notebook executes the training pipeline and exports all fitted model artifacts (TF-IDF vectorizers, encoders, base learners, meta-learner, threshold multipliers) to a compressed `.joblib` file for web app deployment.

## 1. Imports and Setup
---

In [ ]:
import numpy as np
import pandas as pd
import gc
import warnings
import re
import os
import joblib
from datetime import datetime

from scipy.sparse import hstack, csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import lightgbm as lgb
from packaging.version import Version
import sklearn

def _lr(**kwargs):
    if Version(sklearn.__version__) < Version("1.6"):
        kwargs['multi_class'] = 'multinomial'
    return LogisticRegression(**kwargs)

warnings.filterwarnings('ignore')
print('All imports successful.')
print(f'sklearn version : {sklearn.__version__}')
print(f'lightgbm version: {lgb.__version__}')


## 2. Paths & Configuration
---

In [ ]:
train_set_path = '/kaggle/input/comment-category-prediction-challenge/train.csv'
test_set_path = '/kaggle/input/comment-category-prediction-challenge/test.csv'
sample_path = '/kaggle/input/comment-category-prediction-challenge/Sample.csv'

if not os.path.exists(train_set_path):
    train_set_path = '/kaggle/input/competitions/comment-category-prediction-challenge/train.csv'
    test_set_path  = '/kaggle/input/competitions/comment-category-prediction-challenge/test.csv'
    sample_path    = '/kaggle/input/competitions/comment-category-prediction-challenge/Sample.csv'

SEED = 42
N_FOLDS = 4
np.random.seed(SEED)

CLASS_NAMES  = {0: 'Normal', 1: 'Offensive', 2: 'Hate Speech', 3: 'Severe/Violent'}
CLASS_LABELS = ['C-0 Normal', 'C-1 Offensive', 'C-2 Hate Speech', 'C-3 Severe/Violent']

LR_WORD_C = 5.395
LR_CHAR_C = 4.464
NBSVM_C = 2.0
lgb_class_weights = {0: 1.0, 1: 3.5, 2: 1.5, 3: 15.0}

print("Paths and configuration initialized.")


## 3. Data Loading
---

In [ ]:
print("Loading datasets...")
train_raw = pd.read_csv(train_set_path)
test_raw = pd.read_csv(test_set_path)
sample_sub = pd.read_csv(sample_path)

train_raw = train_raw.drop_duplicates(subset=['comment'], keep='first').reset_index(drop=True)
train_labels = train_raw['label'].values.astype(int)
train_features = train_raw.drop(columns=['label'])

print(f"Train samples: {len(train_features):,}")
print(f"Test samples:  {len(test_raw):,}")


## 4. Text Preprocessing & Custom Transformers
---

In [ ]:
LEET_MAP = str.maketrans({
    '0': 'o', '1': 'i', '3': 'e', '4': 'a',
    '5': 's', '7': 't', '@': 'a', '$': 's', '!': 'i'
})

ABBREV_MAP = {
    'u'    : 'you',
    'r'    : 'are',
    'ur'   : 'your',
    'gonna': 'going to',
    'wanna': 'want to',
    'kys'  : 'kill yourself',
    'kms'  : 'kill myself',
    'wtf'  : 'what the fuck',
    'stfu' : 'shut the fuck up',
    'idk'  : 'i do not know',
    'ngl'  : 'not gonna lie',
}

def expand_abbrevs(sentence):
    words = sentence.split()
    result = []
    for w in words:
        if w in ABBREV_MAP:
            result.append(ABBREV_MAP[w])
        else:
            result.append(w)
    return ' '.join(result)

def clean_text_column(df):
    df = df.copy()
    text = df['comment'].fillna('').astype(str)

    text = text.str.lower()

    text = text.str.replace(r'http\S+|www\.\S+', ' ', regex=True)
    text = text.str.replace(r'\w+\.(com|org|net|co|us)', ' ', regex=True)
    text = text.str.replace(r'<[^>]+>', ' ', regex=True)

    text = text.apply(lambda s: s.translate(LEET_MAP))
    text = text.apply(expand_abbrevs)

    text = text.str.replace(r'[^a-z\s]', ' ', regex=True)
    text = text.str.replace(r'\s+', ' ', regex=True).str.strip()

    text = text.replace('', 'empty_comment').fillna('missing_comment')

    df['comment_clean'] = text
    return df

class NBTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y=None):
        if y is None:
            self.r_ = np.ones(X.shape[1], dtype=np.float32)
            return self

        labels = np.unique(y)
        num_cols = X.shape[1]
        ratios = np.zeros((len(labels), num_cols), dtype=np.float64)

        for i, lbl in enumerate(labels):
            mask = (y == lbl)
            p_counts = np.asarray(X[mask].sum(axis=0)).flatten() + self.alpha
            n_counts = np.asarray(X[~mask].sum(axis=0)).flatten() + self.alpha
            
            p_freq = p_counts / p_counts.sum()
            n_freq = n_counts / n_counts.sum()
            ratios[i] = np.log(p_freq) - np.log(n_freq)

        self.r_ = np.abs(ratios).max(axis=0).astype(np.float32)
        return self

    def transform(self, X):
        return X.multiply(self.r_)

print('TextCleaner and NBTransformer ready.')


## 5. Feature Engineering
---

In [ ]:
POSITIVE_WORDS = frozenset({
    'good','great','excellent','amazing','wonderful','best','love','perfect',
    'nice','fantastic','awesome','beautiful','brilliant','outstanding','happy',
    'glad','pleased','thankful','thanks','appreciate','agree','helpful','useful',
    'interesting','impressive','well','better','fine','cool','fun','enjoy',
    'liked','support','positive','fair','kind','respect','right','correct'
})

NEGATIVE_WORDS = frozenset({
    'bad','terrible','awful','hate','worst','horrible','disgusting','poor',
    'pathetic','stupid','ugly','wrong','annoying','boring','dumb','useless',
    'trash','garbage','waste','sucks','lame','crap','fail','failed','worse',
    'angry','mad','upset','disappointed','sad','broken','ruined','ridiculous',
    'offensive','toxic','evil','idiot','fool','shut','die','kill'
})

VIOLENCE_WORDS = frozenset({
    'kill','killed','killing','shoot','shooting','shot','death','dead','die',
    'dying','murder','murdered','murders','attack','attacking','attacked',
    'burn','burning','burned','fire','firing','guns','gun','weapon','weapons',
    'bomb','bombing','destroy','destroying','violence','violent','rape','hang',
    'hanging','decapitate','decapitated','torture','tortured','stab','stabbing',
    'threat','threaten'
})

def count_unique(text):
    return len(set(str(text).split()))

def count_pos(text):
    words = str(text).split()
    count = 0
    for w in words:
        if w in POSITIVE_WORDS:
            count += 1
    return count

def count_neg(text):
    words = str(text).split()
    count = 0
    for w in words:
        if w in NEGATIVE_WORDS:
            count += 1
    return count

def count_caps(text):
    words = str(text).split()
    count = 0
    for w in words:
        if w.isupper() and len(w) > 1:
            count += 1
    return count

def count_violence(text):
    words = str(text).split()
    count = 0
    for w in words:
        if w in VIOLENCE_WORDS:
            count += 1
    return count

def build_features(df):
    df  = df.copy()
    raw = df['comment'].astype(str)
    cln = df.get('comment_clean', raw.str.lower())

    word_count = cln.str.split().str.len().fillna(0).astype(int)

    if1 = df['if_1'].fillna(0)
    if2 = df['if_2'].fillna(0)

    df['char_count'] = cln.str.len().astype(np.float32)
    df['word_count'] = word_count.astype(np.float32)
    df['unique_words'] = cln.apply(count_unique).astype(np.float32)
    df['lexical_div'] = (df['unique_words'] / (word_count + 1)).astype(np.float32)
    df['avg_word_len'] = (df['char_count'] / (word_count + 1)).astype(np.float32)

    df['caps_count'] = raw.str.count(r'[A-Z]').astype(np.float32)
    df['caps_ratio'] = (df['caps_count'] / (df['char_count'] + 1)).astype(np.float32)
    df['exclaim'] = raw.str.count('!').astype(np.float32)
    df['question'] = raw.str.count(r'\?').astype(np.float32)
    df['punct_count'] = raw.str.count(r'[^\w\s]').astype(np.float32)
    df['sent_count'] = raw.str.count(r'[.!?]+').clip(lower=1).astype(np.float32)
    df['avg_sent_len'] = (word_count / (df['sent_count'] + 1)).astype(np.float32)
    df['all_caps_words'] = raw.apply(count_caps).astype(np.float32)

    df['pos_count'] = cln.apply(count_pos).astype(np.float32)
    df['neg_count'] = cln.apply(count_neg).astype(np.float32)
    df['sent_balance'] = (df['pos_count'] - df['neg_count']).astype(np.float32)

    df['violence_count'] = cln.apply(count_violence).astype(np.float32)
    df['has_violence'] = (df['violence_count'] > 0).astype(np.int8)
    df['violence_ratio'] = (df['violence_count'] / (word_count + 1)).astype(np.float32)
    df['violence_score'] = (
        df['violence_count'] * 2.0 + df['violence_ratio'] * 10.0
    ).astype(np.float32)
    df['if1_x_violence'] = (if1 * df['violence_count']).astype(np.float32)

    df['upvote']   = df['upvote'].fillna(0)
    df['downvote'] = df['downvote'].fillna(0)

    df['total_votes'] = (df['upvote'] + df['downvote']).astype(np.float32)
    df['vote_ratio'] = (df['upvote'] / (df['total_votes'] + 1)).astype(np.float32)
    df['zero_downvote'] = (df['downvote'] == 0).astype(np.int8)
    df['controversy'] = np.minimum(df['upvote'], df['downvote']).astype(np.float32) * 2

    emo_cols = ['emoticon_1', 'emoticon_2', 'emoticon_3']
    df[emo_cols] = df[emo_cols].fillna(0)
    df['total_emo'] = df[emo_cols].sum(axis=1).astype(np.float32)
    df['has_emo'] = (df['total_emo'] > 0).astype(np.int8)

    parsed_dt = pd.to_datetime(df['created_date'], errors='coerce')
    hour = parsed_dt.dt.hour.fillna(12).astype(int)
    day_of_week = parsed_dt.dt.dayofweek.fillna(3).astype(int)

    df['hour_sin']= np.sin(2 * np.pi * hour / 24).astype(np.float32)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24).astype(np.float32)
    df['dow_sin'] = np.sin(2 * np.pi * day_of_week / 7).astype(np.float32)
    df['is_weekend'] = (day_of_week >= 5).astype(np.int8)

    race_col = df['race'].fillna('_').str.lower()
    religion_col = df['religion'].fillna('_').str.lower()
    gender_col = df['gender'].fillna('_').str.lower()

    has_race = (race_col     != '_') & (race_col     != 'none')
    has_religion = (religion_col != '_') & (religion_col != 'none')
    has_gender = (gender_col   != '_') & (gender_col   != 'none')

    df['has_identity'] = (has_race | has_religion | has_gender).astype(np.int8)
    df['identity_count'] = (
        has_race.astype(int) + has_religion.astype(int) + has_gender.astype(int)
    ).astype(np.int8)
    df['disability_flag'] = (
        df['disability'].map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0)
    ).astype(np.int8)

    df['if_prod'] = (if1 * if2).astype(np.float32)
    df['if_ratio'] = (if1 / (if2 + 1)).astype(np.float32)
    df['if_sum'] = (if1 + if2).astype(np.float32)

    df['zone_safe'] = (if2 <= 7).astype(np.int8)
    df['zone_c1_trigger'] = ((if2 >= 8) & (if1.isin([4, 6, 10]))).astype(np.int8)
    df['zone_c23'] = ((if1 == 0) & (if2 >= 8)).astype(np.int8)
    df['if1_low'] = (if1 <= 1).astype(np.int8)
    df['if1_nonzero'] = (if1 > 0).astype(np.int8)

    df['golden_c1'] = ((if2 >= 8) & df['has_identity'].astype(bool)).astype(np.int8)
    df['golden_c2'] = ((if2 >= 8) & ~df['has_identity'].astype(bool)).astype(np.int8)

    df['is_short'] = (word_count <= 15).astype(np.int8)
    df['short_violent'] = (df['is_short'] * df['has_violence']).astype(np.int8)

    df['danger_zone'] = ((df['zone_c23'] == 1) & (df['has_identity'] == 0)).astype(np.int8)
    df['violence_in_dz'] = (df['violence_count'] * df['danger_zone']).astype(np.float32)
    df['c3_signal'] = (df['is_short'] * df['zero_downvote'] * (df['has_identity'] == 0) * df['zone_c23']).astype(np.int8)

    numeric_dtype_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_dtype_cols] = df[numeric_dtype_cols].fillna(0)

    return df

print('FeatureEngineer ready.')


## 6. Vectorization & Feature Matrix Assembly
---

In [ ]:
print('Running clean_text_column and build_features...')
train_engineered = build_features(clean_text_column(train_features))
test_engineered = build_features(clean_text_column(test_raw))

train_text = train_engineered['comment_clean'].values
test_text  = test_engineered['comment_clean'].values

print('Building TF-IDF matrices...')
tfidf_word = TfidfVectorizer(
    max_features=35000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.92,
    sublinear_tf=True,
    strip_accents='unicode',
    dtype=np.float32
)

tfidf_char = TfidfVectorizer(
    max_features=10000,
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=3,
    dtype=np.float32
)

tfidf_phrase = TfidfVectorizer(
    max_features=10000,
    ngram_range=(2, 3),
    min_df=3,
    max_df=0.92,
    sublinear_tf=True,
    dtype=np.float32
)

train_tfidf = hstack([
    tfidf_word.fit_transform(train_text),
    tfidf_char.fit_transform(train_text),
    tfidf_phrase.fit_transform(train_text)
])

test_tfidf = hstack([
    tfidf_word.transform(test_text),
    tfidf_char.transform(test_text),
    tfidf_phrase.transform(test_text)
])

EXCLUDE_COLS = {
    'comment', 'comment_clean', 'created_date',
    'post_id', 'race', 'religion', 'gender', 'disability'
}
numeric_col_names = [
    col for col in train_engineered.columns
    if col not in EXCLUDE_COLS
    and train_engineered[col].dtype in [
        np.float64, np.int64, np.float32, np.int32, np.int8
    ]
]

train_num_scaled = train_engineered[numeric_col_names].values.astype(np.float32)
test_num_scaled = test_engineered[numeric_col_names].values.astype(np.float32)

cat_col_names = ['race', 'religion', 'gender']
ordinal_enc = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

train_cats = ordinal_enc.fit_transform(
    train_engineered[cat_col_names].fillna('missing').astype(str)).astype(np.int32)
test_cats = ordinal_enc.transform(
    test_engineered[cat_col_names].fillna('missing').astype(str)).astype(np.int32)

X_train_full = hstack([
    train_tfidf,
    csr_matrix(train_num_scaled),
    csr_matrix(train_cats)
]).tocsr()

X_test_full = hstack([
    test_tfidf,
    csr_matrix(test_num_scaled),
    csr_matrix(test_cats)
]).tocsr()

del train_tfidf, test_tfidf, train_num_scaled, test_num_scaled
del train_engineered, test_engineered
gc.collect()

print(f'X_train_full shape: {X_train_full.shape}')
print(f'X_test_full  shape: {X_test_full.shape}')


## 7. Stratified K-Fold Stacking CV (7 Base Models)
---

In [ ]:
print(f'{N_FOLDS}-FOLD STRATIFIED CV : 7 BASE MODELS')
print('=' * 60)

kfold   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
n_train = X_train_full.shape[0]
n_test  = X_test_full.shape[0]

oof_lgb     = np.zeros((n_train, 4), dtype=np.float32)
oof_lr_w    = np.zeros((n_train, 4), dtype=np.float32)
oof_lr_c    = np.zeros((n_train, 4), dtype=np.float32)
oof_nbsvm   = np.zeros((n_train, 4), dtype=np.float32)
oof_lgb_bin = np.zeros((n_train, 2), dtype=np.float32)
oof_c23     = np.zeros((n_train, 2), dtype=np.float32)
oof_svc     = np.zeros((n_train, 4), dtype=np.float32)

tst_lgb     = np.zeros((n_test, 4), dtype=np.float32)
tst_lr_w    = np.zeros((n_test, 4), dtype=np.float32)
tst_lr_c    = np.zeros((n_test, 4), dtype=np.float32)
tst_nbsvm   = np.zeros((n_test, 4), dtype=np.float32)
tst_lgb_bin = np.zeros((n_test, 2), dtype=np.float32)
tst_c23     = np.zeros((n_test, 2), dtype=np.float32)
tst_svc     = np.zeros((n_test, 4), dtype=np.float32)

LGB_PARAMS = dict(
    objective='multiclass',
    num_class=4,
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=-1,
    max_bin=127,
    colsample_bytree=0.5,
    subsample=0.9,
    subsample_freq=1,
    min_child_samples=10,
    class_weight=lgb_class_weights,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

for fold_num, (train_idx, val_idx) in enumerate(kfold.split(X_train_full, train_labels), 1):
    print(f'\n  FOLD {fold_num}/{N_FOLDS}')

    y_train = train_labels[train_idx]
    y_val   = train_labels[val_idx]

    # [1/7] LightGBM Multiclass
    print('  [1/7] LightGBM — full feature matrix')
    X_tr_lgb = X_train_full[train_idx]
    X_va_lgb = X_train_full[val_idx]
    cat_feature_indices = [
        X_train_full.shape[1] - 3,
        X_train_full.shape[1] - 2,
        X_train_full.shape[1] - 1
    ]
    lgb_model = lgb.LGBMClassifier(**LGB_PARAMS)
    lgb_model.fit(
        X_tr_lgb, y_train,
        eval_set=[(X_va_lgb, y_val)],
        callbacks=[
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(0)
        ],
        categorical_feature=cat_feature_indices
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va_lgb)
    tst_lgb         += lgb_model.predict_proba(X_test_full) / N_FOLDS
    del X_tr_lgb, X_va_lgb, lgb_model
    gc.collect()

    # [2/7] LR-Word
    print('  [2/7] LR-Word — 60k word n-grams')
    lr_word = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=60000,
            ngram_range=(1, 3),
            sublinear_tf=True,
            min_df=2,
            max_df=0.95,
            strip_accents='unicode',
            token_pattern=r'(?u)\b\w+\b',
            dtype=np.float32
        )),
        ('lr', _lr(
            C=LR_WORD_C,
            class_weight='balanced',
            solver='liblinear',
            max_iter=1000,
            dual=False,
            tol=1e-3,
            random_state=SEED
        ))
    ])
    lr_word.fit(train_text[train_idx], y_train)
    oof_lr_w[val_idx] = lr_word.predict_proba(train_text[val_idx])
    tst_lr_w         += lr_word.predict_proba(test_text) / N_FOLDS
    del lr_word
    gc.collect()

    # [3/7] LR-Char
    print('  [3/7] LR-Char — 30k character n-grams')
    lr_char = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=30000,
            analyzer='char',
            ngram_range=(3, 6),
            sublinear_tf=True,
            min_df=5,
            dtype=np.float32
        )),
        ('lr', _lr(
            C=LR_CHAR_C,
            class_weight='balanced',
            solver='lbfgs',
            max_iter=1000,
            random_state=SEED
        ))
    ])
    lr_char.fit(train_text[train_idx], y_train)
    oof_lr_c[val_idx] = lr_char.predict_proba(train_text[val_idx])
    tst_lr_c         += lr_char.predict_proba(test_text) / N_FOLDS
    del lr_char
    gc.collect()

    # [4/7] NB-SVM
    print('  [4/7] NB-SVM — NB feature weighting + LR')
    nbsvm = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=2,
            max_df=0.95,
            strip_accents='unicode',
            dtype=np.float32
        )),
        ('nb',  NBTransformer(alpha=1.0)),
        ('lr',  _lr(
            C=NBSVM_C,
            class_weight='balanced',
            solver='lbfgs',
            max_iter=1000,
            random_state=SEED
        ))
    ])
    nbsvm.fit(train_text[train_idx], y_train)
    oof_nbsvm[val_idx] = nbsvm.predict_proba(train_text[val_idx])
    tst_nbsvm         += nbsvm.predict_proba(test_text) / N_FOLDS
    del nbsvm
    gc.collect()

    # [5/7] LGB-Binary
    print('  [5/7] LGB-Binary — Class 3 vs rest')
    y_train_lgb_bin = (y_train == 3).astype(int)
    y_val_lgb_bin   = (y_val   == 3).astype(int)
    lgb_bin_model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=63,
        max_bin=127,
        colsample_bytree=0.4,
        subsample=0.8,
        subsample_freq=1,
        min_child_samples=20,
        class_weight={0: 1, 1: 20},
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )
    lgb_bin_model.fit(
        X_train_full[train_idx], y_train_lgb_bin,
        eval_set=[(X_train_full[val_idx], y_val_lgb_bin)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    oof_lgb_bin[val_idx] = lgb_bin_model.predict_proba(X_train_full[val_idx])
    tst_lgb_bin         += lgb_bin_model.predict_proba(X_test_full) / N_FOLDS
    del lgb_bin_model
    gc.collect()

    # [6/7] LGB C3 vs C2 Specialist
    print('  [6/7] LGB C3vC2 — Class 3 vs Class 2 specialist')
    c23_train_mask = (y_train == 2) | (y_train == 3)
    c23_val_mask   = (y_val   == 2) | (y_val   == 3)
    y_train_c23 = (y_train[c23_train_mask] == 3).astype(int)
    y_val_c23   = (y_val[c23_val_mask]     == 3).astype(int)
    lgb_c23_model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=63,
        max_bin=127,
        colsample_bytree=0.4,
        subsample=0.8,
        subsample_freq=1,
        min_child_samples=20,
        class_weight={0: 1, 1: 12},
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    )
    lgb_c23_model.fit(
        X_train_full[train_idx][c23_train_mask], y_train_c23,
        eval_set=[(X_train_full[val_idx][c23_val_mask], y_val_c23)],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )
    oof_c23[val_idx] = lgb_c23_model.predict_proba(X_train_full[val_idx])
    tst_c23         += lgb_c23_model.predict_proba(X_test_full) / N_FOLDS
    del lgb_c23_model
    gc.collect()

    # [7/7] LinearSVC (Calibrated)
    print('  [7/7] LinearSVC — calibrated probability output')
    svc_base = LinearSVC(C=0.184, class_weight='balanced', max_iter=1000, random_state=SEED)
    p_svc = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=2,
            max_df=0.95,
            strip_accents='unicode',
            dtype=np.float32
        )),
        ('svc', CalibratedClassifierCV(svc_base, method='sigmoid', cv=2))
    ])
    p_svc.fit(train_text[train_idx], y_train)
    oof_svc[val_idx] = p_svc.predict_proba(train_text[val_idx])
    tst_svc         += p_svc.predict_proba(test_text) / N_FOLDS
    del p_svc
    gc.collect()

print("\nOOF Predictions completed for all 7 base models.")


## 8. Train Full Base Models for Deployment Artifact
---

In [ ]:
print("Training final base models on 100% of training data for artifact export...")

# 1. Final LightGBM
cat_feature_indices = [
    X_train_full.shape[1] - 3,
    X_train_full.shape[1] - 2,
    X_train_full.shape[1] - 1
]
lgb_model_full = lgb.LGBMClassifier(**LGB_PARAMS)
lgb_model_full.fit(X_train_full, train_labels, categorical_feature=cat_feature_indices)

# 2. Final LR-Word
lr_word_full = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=60000, ngram_range=(1, 3), sublinear_tf=True,
        min_df=2, max_df=0.95, strip_accents='unicode',
        token_pattern=r'(?u)\b\w+\b', dtype=np.float32
    )),
    ('lr', _lr(C=LR_WORD_C, class_weight='balanced', solver='liblinear', max_iter=1000, dual=False, tol=1e-3, random_state=SEED))
])
lr_word_full.fit(train_text, train_labels)

# 3. Final LR-Char
lr_char_full = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=30000, analyzer='char', ngram_range=(3, 6),
        sublinear_tf=True, min_df=5, dtype=np.float32
    )),
    ('lr', _lr(C=LR_CHAR_C, class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=SEED))
])
lr_char_full.fit(train_text, train_labels)

# 4. Final NB-SVM
nbsvm_full = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000, ngram_range=(1, 2), sublinear_tf=True,
        min_df=2, max_df=0.95, strip_accents='unicode', dtype=np.float32
    )),
    ('nb', NBTransformer(alpha=1.0)),
    ('lr', _lr(C=NBSVM_C, class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=SEED))
])
nbsvm_full.fit(train_text, train_labels)

# 5. Final LGB-Binary (C3 vs rest)
y_full_bin = (train_labels == 3).astype(int)
lgb_bin_full = lgb.LGBMClassifier(
    objective='binary', n_estimators=500, learning_rate=0.03, num_leaves=63,
    max_bin=127, colsample_bytree=0.4, subsample=0.8, subsample_freq=1,
    min_child_samples=20, class_weight={0: 1, 1: 20}, random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_bin_full.fit(X_train_full, y_full_bin)

# 6. Final LGB-C23 Specialist (C3 vs C2)
c23_mask = (train_labels == 2) | (train_labels == 3)
y_full_c23 = (train_labels[c23_mask] == 3).astype(int)
lgb_c23_full = lgb.LGBMClassifier(
    objective='binary', n_estimators=500, learning_rate=0.03, num_leaves=63,
    max_bin=127, colsample_bytree=0.4, subsample=0.8, subsample_freq=1,
    min_child_samples=20, class_weight={0: 1, 1: 12}, random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_c23_full.fit(X_train_full[c23_mask], y_full_c23)

# 7. Final LinearSVC
svc_base_full = LinearSVC(C=0.184, class_weight='balanced', max_iter=1000, random_state=SEED)
svc_full = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000, ngram_range=(1, 2), sublinear_tf=True,
        min_df=2, max_df=0.95, strip_accents='unicode', dtype=np.float32
    )),
    ('svc', CalibratedClassifierCV(svc_base_full, method='sigmoid', cv=2))
])
svc_full.fit(train_text, train_labels)

print("All full base models successfully trained.")


## 9. Meta-Learner & Threshold Multipliers
---

In [ ]:
print('META-LEARNER + THRESHOLD APPLICATION')
print('=' * 60)

stacked_train = np.hstack([oof_lgb, oof_lr_w, oof_lr_c, oof_nbsvm,
                            oof_lgb_bin, oof_svc, oof_c23]).astype(np.float32)
stacked_test  = np.hstack([tst_lgb, tst_lr_w, tst_lr_c, tst_nbsvm,
                            tst_lgb_bin, tst_svc, tst_c23]).astype(np.float32)

lgb_meta_params = dict(
    objective='multiclass',
    num_class=4,
    num_leaves=16,
    n_estimators=100,
    learning_rate=0.05,
    min_child_samples=200,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

final_meta = lgb.LGBMClassifier(**lgb_meta_params)
final_meta.fit(stacked_train, train_labels)

BEST_MULTS = [1.032, 0.951, 0.85, 1.25]

test_meta_proba = final_meta.predict_proba(stacked_test)
for i in range(4):
    test_meta_proba[:, i] *= BEST_MULTS[i]

final_preds = test_meta_proba.argmax(1)
print(f"Prediction complete. First 10 test predictions: {final_preds[:10]}")


## 10. Export Deployment Artifact (`joblib.dump`)
---

In [ ]:
artifact_path = '/kaggle/working/comment_classifier_pipeline.joblib'
if not os.path.exists('/kaggle/working'):
    artifact_path = 'comment_classifier_pipeline.joblib'

artifact = {
    'tfidf_word': tfidf_word,
    'tfidf_char': tfidf_char,
    'tfidf_phrase': tfidf_phrase,
    'ordinal_enc': ordinal_enc,
    'numeric_col_names': numeric_col_names,
    'cat_col_names': cat_col_names,
    'lgb_model': lgb_model_full,
    'lr_word': lr_word_full,
    'lr_char': lr_char_full,
    'nbsvm_model': nbsvm_full,
    'lgb_bin_model': lgb_bin_full,
    'lgb_c23_model': lgb_c23_full,
    'svc_model': svc_full,
    'meta_model': final_meta,
    'thresholds': BEST_MULTS,
    'class_names': CLASS_NAMES,
}

joblib.dump(artifact, artifact_path, compress=3)
print(f"Saved deployment artifact successfully to '{artifact_path}'. File size: {os.path.getsize(artifact_path) / (1024*1024):.2f} MB")


## 11. Save Submission CSV
---

In [ ]:
submission = sample_sub.copy()
submission['label'] = final_preds
submission.to_csv('submission.csv', index=False)

print('Submission saved to submission.csv')
print('\nTest prediction distribution:')
pred_counts = pd.Series(final_preds).value_counts().sort_index()
for cls_id, count in pred_counts.items():
    pct = count / len(final_preds) * 100
    print(f'  Class {cls_id} ({CLASS_NAMES[cls_id]:14s}): {count:6,} ({pct:.2f}%)')


## 12. Inference Demonstration for Web App Deployment
---

In [ ]:
def predict_comment(df_raw, model_artifact):
    """
    Single or batch inference function for live web applications (Gradio / FastAPI).
    df_raw must be a pandas DataFrame containing comment text and tabular columns.
    """
    df_clean = clean_text_column(df_raw)
    df_eng = build_features(df_clean)
    clean_text = df_eng['comment_clean'].values
    
    w_vec = model_artifact['tfidf_word'].transform(clean_text)
    c_vec = model_artifact['tfidf_char'].transform(clean_text)
    p_vec = model_artifact['tfidf_phrase'].transform(clean_text)
    
    num_feats = df_eng[model_artifact['numeric_col_names']].values.astype(np.float32)
    cat_feats = model_artifact['ordinal_enc'].transform(
        df_eng[model_artifact['cat_col_names']].fillna('missing').astype(str)
    ).astype(np.int32)
    
    X_full = hstack([w_vec, c_vec, p_vec, csr_matrix(num_feats), csr_matrix(cat_feats)]).tocsr()
    
    p_lgb = model_artifact['lgb_model'].predict_proba(X_full)
    p_lr_w = model_artifact['lr_word'].predict_proba(clean_text)
    p_lr_c = model_artifact['lr_char'].predict_proba(clean_text)
    p_nbsvm = model_artifact['nbsvm_model'].predict_proba(clean_text)
    p_lgb_bin = model_artifact['lgb_bin_model'].predict_proba(X_full)
    p_svc = model_artifact['svc_model'].predict_proba(clean_text)
    p_c23 = model_artifact['lgb_c23_model'].predict_proba(X_full)
    
    stacked = np.hstack([p_lgb, p_lr_w, p_lr_c, p_nbsvm, p_lgb_bin, p_svc, p_c23]).astype(np.float32)
    
    meta_probs = model_artifact['meta_model'].predict_proba(stacked)
    mults = model_artifact['thresholds']
    for idx in range(4):
        meta_probs[:, idx] *= mults[idx]
        
    preds = meta_probs.argmax(axis=1)
    return preds, meta_probs

print("Inference function 'predict_comment' initialized.")
